# Fine-Tuning IndoBERT untuk Klasifikasi Aspek Multi-Label (Ulasan Aplikasi)
Latih model pre-trained IndoBERT (indobenchmark/indobert-base-p1) pada klasifikasi aspek multi-label menggunakan Binary Cross-Entropy Loss.

In [1]:
!pip install transformers accelerate datasets scikit-learn -q

In [2]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset, Features, Sequence, Value
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
import json

## 1. Load Data

In [3]:
dataset_path = 'cleaned_app_reviews_with_aspect_labels.csv'
df = pd.read_csv(dataset_path).dropna(subset=['text_clean'])
print(f"Total data: {len(df)} baris")

Total data: 4885 baris


## 2. Format Input & Targets

In [4]:
aspect_cols = ['aspek_harga', 'aspek_aplikasi', 'aspek_pelayanan', 'aspek_kecepatan', 'aspek_stok', 'aspek_pengiriman', 'aspek_pesanan']
df['labels'] = df[aspect_cols].values.tolist()

print("Contoh input ulasan:")
print(df['text_clean'].iloc[0])
print("Contoh target labels:")
print(df['labels'].iloc[0])

Contoh input ulasan:
kopi bikin melek maleman
Contoh target labels:
[0, 0, 0, 0, 0, 0, 0]


## 3. Inisialisasi Tokenizer

In [5]:
model_name = 'indobenchmark/indobert-base-p1'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text_clean'], padding='max_length', truncation=True, max_length=128)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## 4. Split Train / Test & Konfigurasi Dataset

In [6]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Definisikan skema data untuk target float (kebutuhan BCE loss)
class_features = Features({
    'text_clean': Value('string'),
    'labels': Sequence(Value('float32'))
})

train_dataset = Dataset.from_pandas(train_df[['text_clean', 'labels']].reset_index(drop=True), features=class_features)
test_dataset = Dataset.from_pandas(test_df[['text_clean', 'labels']].reset_index(drop=True), features=class_features)

train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/3908 [00:00<?, ? examples/s]

Map:   0%|          | 0/977 [00:00<?, ? examples/s]

## 5. Konfigurasi Model Multi-Label

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(aspect_cols),
    problem_type="multi_label_classification"
)

[transformers] You passed `num_labels=7` which is incompatible to the `id2label` map of length `5`.


pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 6. Metrik Evaluasi

In [8]:
def multi_label_metrics(predictions, labels, threshold=0.5):
    sigmoid = 1 / (1 + np.exp(-predictions))
    y_pred = np.zeros(sigmoid.shape)
    y_pred[sigmoid >= threshold] = 1
    y_true = labels

    f1_micro_average = f1_score(y_true=y_true, y_pred=y_pred, average='micro')
    f1_macro_average = f1_score(y_true=y_true, y_pred=y_pred, average='macro')
    roc_auc = roc_auc_score(y_true, sigmoid, average = 'micro')
    accuracy = accuracy_score(y_true, y_pred)

    metrics = {
        'f1_micro': f1_micro_average,
        'f1_macro': f1_macro_average,
        'roc_auc': roc_auc,
        'accuracy': accuracy
    }
    return metrics

def compute_metrics(p):
    predictions, labels = p
    return multi_label_metrics(predictions=predictions, labels=labels)

## 7. Setup Training Arguments & Trainer

In [9]:
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_dir='./logs',
    logging_steps=50,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## 8. Eksekusi Training

In [10]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Roc Auc,Accuracy
1,0.068604,0.055505,0.916415,0.805087,0.987875,0.933470


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Roc Auc,Accuracy
1,0.068604,0.055505,0.916415,0.805087,0.987875,0.933470
2,0.038218,0.036995,0.949153,0.839624,0.993920,0.953941
3,0.024552,0.029034,0.961652,0.922005,0.995001,0.962129
4,0.015545,0.024298,0.967423,0.943382,0.996070,0.967247
5,0.013555,0.023470,0.968442,0.946494,0.996014,0.969294


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1225, training_loss=0.04361623586440573, metrics={'train_runtime': 530.4795, 'train_samples_per_second': 36.835, 'train_steps_per_second': 2.309, 'total_flos': 1285355206272000.0, 'train_loss': 0.04361623586440573, 'epoch': 5.0})

## 9. Simpan Model

In [11]:
model_save_path = 'weights/indobert_aspect_multilabel'
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print("Notebook setup selesai. Siap dijalankan di GPU untuk fine-tuning.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Notebook setup selesai. Siap dijalankan di GPU untuk fine-tuning.


In [13]:
import shutil
from google.colab import files

# Buat file ZIP
shutil.make_archive(
    "indobert_aspect_multilabel",
    "zip",
    "weights/indobert_aspect_multilabel"
)

# Download otomatis
files.download("indobert_aspect_multilabel.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>